# F6-svd-spectral — Practice p20 — Solution


A fractional float cannot be transmitted, so the 15% allowance is floored to 460 whole floats before ranks and slack are computed.


In [ ]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)
strengths = np.array([90., 35., 18., 9., 4., 2.])
Uraw = rng.normal(0, 1, (64, 6))
Vraw = rng.normal(0, 1, (48, 6))
Un = Uraw / np.sqrt((Uraw**2).sum(axis=0))
Vn = Vraw / np.sqrt((Vraw**2).sum(axis=0))
W = (Un * strengths) @ Vn.T + 0.08 * rng.normal(0, 1, (64, 48))

floats_full = 64 * 48
budget_floats = int(0.15 * floats_full)
cost_per_rank = 64 + 48 + 1
r_max = budget_floats // cost_per_rank
s = np.linalg.svd(W, full_matrices=False, compute_uv=False)
fro = np.linalg.norm(W)
rel = np.array([np.sqrt((s[r:]**2).sum()) / fro for r in range(49)])
rel_at_rmax = rel[r_max]
quality_ok = bool(rel_at_rmax <= 0.08)
qualifies = rel <= 0.08
assert qualifies.any()
r_min_quality = int(np.argmax(qualifies))
slack_floats = max(0, budget_floats - r_min_quality * cost_per_rank)
r_max, rel_at_rmax, quality_ok, r_min_quality, slack_floats


Send rank $4$: it costs $452$ floats, within the $460$-float whole-number budget, leaving $8$ floats unused. Its relative Frobenius error is about $5.90\%$, below the $8\%$ quality bar, and rank $4$ is the smallest rank meeting that bar.


### Answer check


In [ ]:
assert budget_floats == 460 and r_max == 4
assert np.isclose(rel_at_rmax, 0.05899200093664548, rtol=0, atol=1e-12)
assert quality_ok is True
assert r_min_quality == 4 and slack_floats == 8
